# OmniAssist AI

# Supervised Fine-Tuning (SFT)

This notebook demonstrates the Supervised Fine-Tuning (SFT) stage of the Customer Support Assistant.

The objective of SFT is to adapt the base language model to follow customer support instructions using an instruction-response dataset.

# Training Pipeline

```
Instruction Dataset

        │

        ▼

Load Base Model

        │

        ▼

Apply QLoRA

        │

        ▼

Configure LoRA

        │

        ▼

Train using SFTTrainer

        │

        ▼

Save LoRA Adapter
```

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(project_root)

In [ ]:
from src.config.model_config import MODEL_NAME
from src.config.training_config import OUTPUT_DIR

print("=" * 60)
print("Base Model")
print("=" * 60)
print(MODEL_NAME)

print()

print("=" * 60)
print("Output Directory")
print("=" * 60)
print(OUTPUT_DIR)

# LoRA Configuration

The project uses Parameter Efficient Fine-Tuning (PEFT) with LoRA.

Only a small number of trainable adapter parameters are optimized while the base model remains frozen.

This significantly reduces GPU memory usage compared to full fine-tuning.

In [ ]:
from src.config.training_config import *

print("=" * 60)
print("Training Configuration")
print("=" * 60)

print(f"Epochs               : {NUM_EPOCHS}")
print(f"Batch Size           : {TRAIN_BATCH_SIZE}")
print(f"Training Subset Size : {TRAIN_SUBSET_SIZE}")  # ----> None for full instruction finetuning and any number for training the subset of total samples...
print(f"Learning Rate        : {LEARNING_RATE}")
print(f"LoRA Rank            : {LORA_R}")
print(f"LoRA Alpha           : {LORA_ALPHA}")
print(f"LoRA Dropout         : {LORA_DROPOUT}")

# Training Configuration Explanation

The Supervised Fine-Tuning stage was executed on the **Google Colab Free Tier (T4 GPU)**.

To ensure successful training within the available GPU memory and runtime limits, the following configuration was adopted.

| Parameter | Value | Purpose |
|-----------|------:|---------|
| Batch Size | 2 | Number of samples processed in each training step |
| Training Subset Size | 600 | Number of instruction samples used during SFT |
| Quantization | 4-bit NF4 | Reduces GPU memory usage |
| Fine-Tuning Method | QLoRA | Parameter-efficient fine-tuning |
| Adapter | LoRA | Trains only lightweight adapter weights |

Using a subset of the dataset significantly reduced training time while still demonstrating the complete Supervised Fine-Tuning workflow.

# Start Training

The complete Supervised Fine-Tuning pipeline is implemented as a reusable Python module.

Executing the following command will

- Load the base model
- Load the instruction dataset
- Apply QLoRA
- Configure LoRA adapters
- Train the model using SFTTrainer
- Save the trained adapter

The notebook calls the project implementation directly instead of duplicating the training code.

In [ ]:
!python -m src.training.train_sft

# Generated Adapter

After training completes, the LoRA adapter is stored inside

```
adapters/

└── sft/

    └── customer_support/
```

In [ ]:
from pathlib import Path

adapter_path = Path("adapters/sft/customer_support")

print("=" * 60)
print("Generated Adapter Files")
print("=" * 60)

if adapter_path.exists():

    for file in adapter_path.rglob("*"):

        print(file)

else:

    print("Adapter directory not found.")

# Summary

The Supervised Fine-Tuning stage successfully adapts the **Qwen2.5-1.5B-Instruct** base model for the Customer Support domain using **QLoRA**.

Key outcomes of this stage include:

- Instruction-following behavior specialized for customer support.
- Parameter-efficient training using LoRA adapters.
- Reduced GPU memory consumption through 4-bit quantization.
- Faster training suitable for Google Colab Free Tier.

The generated SFT adapter serves as the initialization point for the subsequent **Direct Preference Optimization (DPO)** stage.